# SmartBasket AI
## Intelligent Shopping & Market Basket Recommendation System

### Notebook 04 — Market Basket Analysis

**Dataset:** Instacart Online Grocery Basket Dataset

---

### Objective

The objective of this notebook is to identify frequently purchased
product combinations and generate association rules that can be used
by the SmartBasket AI recommendation engine.

### Key Tasks

1. Prepare transaction baskets
2. Analyze product combinations
3. Generate frequent itemsets
4. Apply FP-Growth
5. Generate association rules
6. Evaluate rules using Support, Confidence, and Lift
7. Identify strong product associations
8. Save the final association rules

# Step 1 Import Libraries

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from collections import Counter
from itertools import combinations

import gc
import time

print("Libraries imported successfully.")

Libraries imported successfully.


# Step 2 - Project Paths

In [2]:
PROJECT_ROOT = (
    Path.cwd()
    .resolve()
    .parent
)

# If notebook is opened from notebooks folder
if PROJECT_ROOT.name != "SmartBasket-AI":
    PROJECT_ROOT = Path.cwd().resolve().parent

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"

PROCESSED_DIR = DATA_DIR / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

prior_path = (
    RAW_DIR /
    "order_products__prior.csv"
)

products_path = (
    RAW_DIR /
    "products.csv"
)

print("Project root:", PROJECT_ROOT)
print("Prior dataset exists:", prior_path.exists())
print("Products dataset exists:", products_path.exists())

Project root: C:\Users\hp\OneDrive\Desktop\SmartBasket-AI
Prior dataset exists: True
Products dataset exists: True


# Step 3 — Load Product Information

In [3]:
products = pd.read_csv(
    products_path,
    usecols=[
        "product_id",
        "product_name",
        "aisle_id",
        "department_id"
    ],
    dtype={
        "product_id": "int32",
        "aisle_id": "int16",
        "department_id": "int16"
    }
)

print(
    "Products:",
    products.shape
)

products.head()

Products: (49688, 4)


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


# Step 4 — Configuration

In [4]:
CHUNK_SIZE = 500_000

# Only most frequently purchased products
TOP_N_PRODUCTS = 1500

# Ignore extremely large baskets
MAX_BASKET_SIZE = 30

# Minimum number of times two products
# must occur together
MIN_PAIR_COUNT = 50

print("Configuration loaded.")
print("Top products:", TOP_N_PRODUCTS)
print("Chunk size:", CHUNK_SIZE)
print("Maximum basket size:", MAX_BASKET_SIZE)
print("Minimum pair count:", MIN_PAIR_COUNT)

Configuration loaded.
Top products: 1500
Chunk size: 500000
Maximum basket size: 30
Minimum pair count: 50


# Step 5 — First Pass: Product Frequency

In [5]:
product_counter = Counter()

start_time = time.time()

for chunk_number, chunk in enumerate(
    pd.read_csv(
        prior_path,
        usecols=[
            "product_id"
        ],
        dtype={
            "product_id": "int32"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    counts = (
        chunk["product_id"]
        .value_counts()
    )

    product_counter.update(
        counts.to_dict()
    )

    if chunk_number % 5 == 0:

        print(
            f"Processed chunks: {chunk_number}"
        )

    del chunk

    gc.collect()


print(
    "\nProduct frequency calculation completed."
)

print(
    "Unique products:",
    len(product_counter)
)

print(
    "Time taken:",
    round(
        time.time() - start_time,
        2
    ),
    "seconds"
)

Processed chunks: 5
Processed chunks: 10
Processed chunks: 15
Processed chunks: 20
Processed chunks: 25
Processed chunks: 30
Processed chunks: 35
Processed chunks: 40
Processed chunks: 45
Processed chunks: 50
Processed chunks: 55
Processed chunks: 60
Processed chunks: 65

Product frequency calculation completed.
Unique products: 49677
Time taken: 14.32 seconds


# Step 6 — Select Top Products

In [6]:
top_products = (
    pd.DataFrame(
        product_counter.items(),
        columns=[
            "product_id",
            "purchase_count"
        ]
    )
    .sort_values(
        "purchase_count",
        ascending=False
    )
    .head(
        TOP_N_PRODUCTS
    )
    .reset_index(drop=True)
)

print(
    "Selected products:",
    len(top_products)
)

top_products.head(20)

Selected products: 1500


,product_id,purchase_count
0,24852,472565
1,13176,379450
2,21137,264683
3,21903,241921
4,47209,213584
5,47766,176815
6,47626,152657
7,16797,142951
8,26209,140627
9,27845,137905


# Step 7 — Create product_frequency.csv

In [7]:
product_frequency = (
    top_products
    .merge(
        products[
            [
                "product_id",
                "product_name",
                "aisle_id",
                "department_id"
            ]
        ],
        on="product_id",
        how="left"
    )
)

product_frequency = (
    product_frequency[
        [
            "product_id",
            "product_name",
            "purchase_count",
            "aisle_id",
            "department_id"
        ]
    ]
)

frequency_output = (
    PROCESSED_DIR /
    "product_frequency.csv"
)

product_frequency.to_csv(
    frequency_output,
    index=False
)

print(
    "Product frequency saved:"
)

print(
    frequency_output
)

product_frequency.head(20)

Product frequency saved:
C:\Users\hp\OneDrive\Desktop\SmartBasket-AI\data\processed\product_frequency.csv


,product_id,product_name,purchase_count,aisle_id,department_id
0,24852,Banana,472565,24,4
1,13176,Bag of Organic Bananas,379450,24,4
2,21137,Organic Strawberries,264683,24,4
3,21903,Organic Baby Spinach,241921,123,4
4,47209,Organic Hass Avocado,213584,24,4
5,47766,Organic Avocado,176815,24,4
6,47626,Large Lemon,152657,24,4
7,16797,Strawberries,142951,24,4
8,26209,Limes,140627,24,4
9,27845,Organic Whole Milk,137905,84,16


# Step 8 — Prepare Top Product Set

In [8]:
top_product_set = set(
    top_products[
        "product_id"
    ].tolist()
)

print(
    "Top product set size:",
    len(top_product_set)
)

Top product set size: 1500


# Step 9 — Generate Product Co-occurrence

In [9]:
pair_counter = Counter()
basket_counter = Counter()

current_order_id = None
current_basket = []

start_time = time.time()


def process_basket(
    basket,
    pair_counter,
    basket_counter
):

    # Remove duplicate products
    basket = list(
        set(basket)
    )

    # Keep only selected frequent products
    basket = [
        product_id
        for product_id in basket
        if product_id in top_product_set
    ]

    # Ignore baskets that are too small
    if len(basket) < 2:
        return

    # Ignore unusually large baskets
    if len(basket) > MAX_BASKET_SIZE:
        return

    basket = sorted(
        basket
    )

    # Count product occurrence
    for product_id in basket:

        basket_counter[
            product_id
        ] += 1

    # Generate product pairs
    for pair in combinations(
        basket,
        2
    ):

        pair_counter[pair] += 1


for chunk_number, chunk in enumerate(
    pd.read_csv(
        prior_path,
        usecols=[
            "order_id",
            "product_id"
        ],
        dtype={
            "order_id": "int32",
            "product_id": "int32"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):

    for order_id, group in chunk.groupby(
        "order_id"
    ):

        product_ids = (
            group["product_id"]
            .tolist()
        )

        process_basket(
            product_ids,
            pair_counter,
            basket_counter
        )


    if chunk_number % 5 == 0:

        print(
            f"Processed chunks: {chunk_number}"
        )

    del chunk

    gc.collect()


print(
    "\nCo-occurrence analysis completed."
)

print(
    "Unique product pairs:",
    len(pair_counter)
)

print(
    "Time taken:",
    round(
        time.time() - start_time,
        2
    ),
    "seconds"
)

Processed chunks: 5
Processed chunks: 10
Processed chunks: 15
Processed chunks: 20
Processed chunks: 25
Processed chunks: 30
Processed chunks: 35
Processed chunks: 40
Processed chunks: 45
Processed chunks: 50
Processed chunks: 55
Processed chunks: 60
Processed chunks: 65

Co-occurrence analysis completed.
Unique product pairs: 1030837
Time taken: 408.9 seconds


# Step 10 - Create Association Rules

In [10]:
total_orders = sum(
    basket_counter.values()
) / max(
    len(basket_counter),
    1
)

# Better estimate of transaction count
# from order_products dataset

order_ids_seen = set()

# Step 11 — Correct Total Orders

In [11]:
total_orders = 0

for chunk in pd.read_csv(
    prior_path,
    usecols=[
        "order_id"
    ],
    dtype={
        "order_id": "int32"
    },
    chunksize=CHUNK_SIZE
):

    total_orders += (
        chunk["order_id"]
        .nunique()
    )

    del chunk

    gc.collect()


print(
    "Total prior orders:",
    total_orders
)

Total prior orders: 3214931


# Step 12 — Calculate Support, Confidence & Lift

In [12]:
rules = []

for (
    (product_a, product_b),
    pair_count
) in pair_counter.items():

    if pair_count < MIN_PAIR_COUNT:
        continue

    count_a = basket_counter.get(
        product_a,
        0
    )

    count_b = basket_counter.get(
        product_b,
        0
    )

    if count_a == 0 or count_b == 0:
        continue

    # A -> B
   

    support_ab = (
        pair_count /
        total_orders
    )

    confidence_ab = (
        pair_count /
        count_a
    )

    support_b = (
        count_b /
        total_orders
    )

    lift_ab = (
        confidence_ab /
        support_b
    )


    rules.append(
        {
            "antecedent_id": product_a,
            "consequent_id": product_b,
            "support": support_ab,
            "confidence": confidence_ab,
            "lift": lift_ab,
            "pair_count": pair_count
        }
    )


    # B -> A

    confidence_ba = (
        pair_count /
        count_b
    )

    support_a = (
        count_a /
        total_orders
    )

    lift_ba = (
        confidence_ba /
        support_a
    )


    rules.append(
        {
            "antecedent_id": product_b,
            "consequent_id": product_a,
            "support": support_ab,
            "confidence": confidence_ba,
            "lift": lift_ba,
            "pair_count": pair_count
        }
    )


association_rules = pd.DataFrame(
    rules
)

print(
    "Generated rules:",
    association_rules.shape
)

association_rules.head()

Generated rules: (659214, 6)


,antecedent_id,consequent_id,support,confidence,lift,pair_count
0,9327,17794,0.000074,0.038725,1.736786,237
1,17794,9327,0.000074,0.003306,1.736786,237
2,9327,28985,0.000067,0.034967,1.696331,214
3,28985,9327,0.000067,0.003229,1.696331,214
4,9327,33120,0.000017,0.008824,1.496468,54


# Step 13 — Filter Strong Rules

In [13]:
MIN_CONFIDENCE = 0.05
MIN_LIFT = 1.10

strong_rules = (
    association_rules[
        (
            association_rules[
                "confidence"
            ]
            >= MIN_CONFIDENCE
        )
        &
        (
            association_rules[
                "lift"
            ]
            >= MIN_LIFT
        )
    ]
    .copy()
)

print(
    "Strong rules:",
    strong_rules.shape
)

strong_rules.head(20)

Strong rules: (27636, 6)


,antecedent_id,consequent_id,support,confidence,lift,pair_count
6,17794,28985,0.001540,0.069053,3.349894,4950
7,28985,17794,0.001540,0.074693,3.349894,4950
12,17461,21903,0.000927,0.180896,2.455888,2979
23,24838,21903,0.002132,0.142064,1.928687,6854
25,32665,21903,0.000220,0.146650,1.990954,707
27,33754,21903,0.000842,0.092314,1.253275,2706
29,46667,21903,0.003198,0.201858,2.740463,10280
60,6184,13176,0.002323,0.258729,2.250722,7469
70,6348,13176,0.000897,0.192395,1.673670,2884
74,6348,27966,0.000526,0.112875,2.703203,1692


# Step 14 — Add Product Names

In [14]:
product_name_map = (
    products
    .set_index(
        "product_id"
    )[
        "product_name"
    ]
    .to_dict()
)


strong_rules[
    "antecedent"
] = (
    strong_rules[
        "antecedent_id"
    ]
    .map(
        product_name_map
    )
)


strong_rules[
    "consequent"
] = (
    strong_rules[
        "consequent_id"
    ]
    .map(
        product_name_map
    )
)


strong_rules = (
    strong_rules[
        [
            "antecedent_id",
            "antecedent",
            "consequent_id",
            "consequent",
            "support",
            "confidence",
            "lift",
            "pair_count"
        ]
    ]
)


print(
    "Product names added."
)

strong_rules.head(20)

Product names added.


,antecedent_id,antecedent,consequent_id,consequent,support,confidence,lift,pair_count
6,17794,Carrots,28985,Michigan Organic Kale,0.001540,0.069053,3.349894,4950
7,28985,Michigan Organic Kale,17794,Carrots,0.001540,0.074693,3.349894,4950
12,17461,Air Chilled Organic Boneless Skinless Chicken ...,21903,Organic Baby Spinach,0.000927,0.180896,2.455888,2979
23,24838,Unsweetened Almondmilk,21903,Organic Baby Spinach,0.002132,0.142064,1.928687,6854
25,32665,Organic Ezekiel 49 Bread Cinnamon Raisin,21903,Organic Baby Spinach,0.000220,0.146650,1.990954,707
27,33754,Total 2% with Strawberry Lowfat Greek Strained...,21903,Organic Baby Spinach,0.000842,0.092314,1.253275,2706
29,46667,Organic Ginger Root,21903,Organic Baby Spinach,0.003198,0.201858,2.740463,10280
60,6184,Clementines,13176,Bag of Organic Bananas,0.002323,0.258729,2.250722,7469
70,6348,Mini Original Babybel Cheese,13176,Bag of Organic Bananas,0.000897,0.192395,1.673670,2884
74,6348,Mini Original Babybel Cheese,27966,Organic Raspberries,0.000526,0.112875,2.703203,1692


# Step 15 — Sort Rules

In [15]:
strong_rules = (
    strong_rules
    .sort_values(
        [
            "lift",
            "confidence",
            "support"
        ],
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

print(
    "Rules sorted successfully."
)

strong_rules.head(20)

Rules sorted successfully.


,antecedent_id,antecedent,consequent_id,consequent,support,confidence,lift,pair_count
0,48220,Almond Milk Peach Yogurt,15984,Almond Milk Blueberry Yogurt,0.000689,0.477380,331.836794,2216
1,15984,Almond Milk Blueberry Yogurt,48220,Almond Milk Peach Yogurt,0.000689,0.479135,331.836794,2216
2,38312,Almond Milk Strawberry Yogurt,15984,Almond Milk Blueberry Yogurt,0.000821,0.469929,326.657027,2641
3,15984,Almond Milk Blueberry Yogurt,38312,Almond Milk Strawberry Yogurt,0.000821,0.571027,326.657027,2641
4,48220,Almond Milk Peach Yogurt,38312,Almond Milk Strawberry Yogurt,0.000758,0.525205,300.444256,2438
5,38312,Almond Milk Strawberry Yogurt,48220,Almond Milk Peach Yogurt,0.000758,0.433808,300.444256,2438
6,23291,Coconut Chia Bar,35633,Chocolate Peanut Butter,0.000573,0.404036,283.798946,1842
7,35633,Chocolate Peanut Butter,23291,Coconut Chia Bar,0.000573,0.402447,283.798946,1842
8,31288,Lowfat Key Lime Yogurt,11737,Organic Lemon Lowfat Yogurt,0.000349,0.296072,248.006806,1123
9,11737,Organic Lemon Lowfat Yogurt,31288,Lowfat Key Lime Yogurt,0.000349,0.292600,248.006806,1123


# Step 16 — Check Example Recommendations

In [16]:
example_products = [
    "Sugar",
    "Rice",
    "Noodles",
    "Bread",
    "Organic Whole Milk"
]


for product in example_products:

    print(
        "\n================================"
    )

    print(
        f"Recommendations for: {product}"
    )

    result = (
        strong_rules[
            strong_rules[
                "antecedent"
            ]
            .str.lower()
            .eq(
                product.lower()
            )
        ]
        [
            [
                "antecedent",
                "consequent",
                "confidence",
                "lift"
            ]
        ]
        .head(10)
    )

    if result.empty:

        print(
            "No direct association rule found."
        )

    else:

        display(
            result
        )


Recommendations for: Sugar
No direct association rule found.

Recommendations for: Rice
No direct association rule found.

Recommendations for: Noodles
No direct association rule found.

Recommendations for: Bread
No direct association rule found.

Recommendations for: Organic Whole Milk


,antecedent,consequent,confidence,lift
5646,Organic Whole Milk,Organic Whole String Cheese,0.063832,3.507040
15018,Organic Whole Milk,Organic Strawberries,0.174019,2.157590
19416,Organic Whole Milk,Organic Garlic,0.060595,1.807763
19532,Organic Whole Milk,Organic Raspberries,0.075168,1.800172
20063,Organic Whole Milk,Organic Avocado,0.095176,1.759408
21031,Organic Whole Milk,Bag of Organic Bananas,0.193915,1.686895
21046,Organic Whole Milk,Organic Yellow Onion,0.058375,1.685709
21205,Organic Whole Milk,Organic Baby Spinach,0.123298,1.673922
21384,Organic Whole Milk,Organic Zucchini,0.053200,1.662044
21541,Organic Whole Milk,Organic Hass Avocado,0.107656,1.651700


# Step 17 — Search Any Product

In [17]:
def show_recommendations(
    product_name,
    top_n=10
):

    result = (
        strong_rules[
            strong_rules[
                "antecedent"
            ]
            .str.lower()
            .eq(
                product_name.lower()
            )
        ]
        [
            [
                "antecedent",
                "consequent",
                "support",
                "confidence",
                "lift",
                "pair_count"
            ]
        ]
        .head(top_n)
    )

    print(
        f"\nRecommendations for: {product_name}"
    )

    if result.empty:

        print(
            "No association rules found."
        )

    else:

        display(
            result
        )


show_recommendations(
    "Sugar",
    top_n=10
)


Recommendations for: Sugar
No association rules found.


In [18]:
show_recommendations(
    "Rice",
    top_n=10
)


Recommendations for: Rice
No association rules found.


In [19]:
show_recommendations(
    "Noodles",
    top_n=10
)


Recommendations for: Noodles
No association rules found.


# Step 18 — Save Final Rules

In [20]:
rules_output = (
    PROCESSED_DIR /
    "association_rules.csv"
)

strong_rules.to_csv(
    rules_output,
    index=False
)

print(
    "Association rules saved successfully:"
)

print(
    rules_output
)

print(
    "Total saved rules:",
    len(strong_rules)
)

Association rules saved successfully:
C:\Users\hp\OneDrive\Desktop\SmartBasket-AI\data\processed\association_rules.csv
Total saved rules: 27636


# Step 19 — Final Validation

In [21]:
print(
    "=========================================="
)

print(
    "SMARTBASKET AI MARKET BASKET ANALYSIS"
)

print(
    "=========================================="
)

print(
    "Total products:",
    len(products)
)

print(
    "Top products analyzed:",
    len(top_products)
)

print(
    "Product pairs:",
    len(pair_counter)
)

print(
    "Association rules:",
    len(association_rules)
)

print(
    "Strong rules:",
    len(strong_rules)
)

print(
    "Output file exists:",
    rules_output.exists()
)

print(
    "Output file:",
    rules_output
)

SMARTBASKET AI MARKET BASKET ANALYSIS
Total products: 49688
Top products analyzed: 1500
Product pairs: 1030837
Association rules: 659214
Strong rules: 27636
Output file exists: True
Output file: C:\Users\hp\OneDrive\Desktop\SmartBasket-AI\data\processed\association_rules.csv


# Step 20 - Verify the aassociation_rules.csv

In [22]:
rules_file = PROCESSED_DIR / "association_rules.csv"

print("File exists:", rules_file.exists())

if rules_file.exists():

    saved_rules = pd.read_csv(
        rules_file
    )

    print(
        "Rules shape:",
        saved_rules.shape
    )

    print(
        "\nColumns:"
    )

    print(
        saved_rules.columns.tolist()
    )

    print(
        "\nFirst 20 rules:"
    )

    display(
        saved_rules.head(20)
    )

File exists: True
Rules shape: (27636, 8)

Columns:
['antecedent_id', 'antecedent', 'consequent_id', 'consequent', 'support', 'confidence', 'lift', 'pair_count']

First 20 rules:


,antecedent_id,antecedent,consequent_id,consequent,support,confidence,lift,pair_count
0,48220,Almond Milk Peach Yogurt,15984,Almond Milk Blueberry Yogurt,0.000689,0.477380,331.836794,2216
1,15984,Almond Milk Blueberry Yogurt,48220,Almond Milk Peach Yogurt,0.000689,0.479135,331.836794,2216
2,38312,Almond Milk Strawberry Yogurt,15984,Almond Milk Blueberry Yogurt,0.000821,0.469929,326.657027,2641
3,15984,Almond Milk Blueberry Yogurt,38312,Almond Milk Strawberry Yogurt,0.000821,0.571027,326.657027,2641
4,48220,Almond Milk Peach Yogurt,38312,Almond Milk Strawberry Yogurt,0.000758,0.525205,300.444256,2438
5,38312,Almond Milk Strawberry Yogurt,48220,Almond Milk Peach Yogurt,0.000758,0.433808,300.444256,2438
6,23291,Coconut Chia Bar,35633,Chocolate Peanut Butter,0.000573,0.404036,283.798946,1842
7,35633,Chocolate Peanut Butter,23291,Coconut Chia Bar,0.000573,0.402447,283.798946,1842
8,31288,Lowfat Key Lime Yogurt,11737,Organic Lemon Lowfat Yogurt,0.000349,0.296072,248.006806,1123
9,11737,Organic Lemon Lowfat Yogurt,31288,Lowfat Key Lime Yogurt,0.000349,0.292600,248.006806,1123


# Step 21 — Sugar / Rice / Noodles test

In [23]:
test_products = [
    "Sugar",
    "Rice",
    "Noodles"
]

for product in test_products:

    print("\n" + "=" * 60)
    print(f"Recommendations for: {product}")
    print("=" * 60)

    result = (
        strong_rules[
            strong_rules["antecedent"]
            .str.lower()
            .eq(product.lower())
        ]
        [
            [
                "antecedent",
                "consequent",
                "support",
                "confidence",
                "lift"
            ]
        ]
        .head(10)
    )

    if result.empty:

        print(
            f"No direct rule found for {product}"
        )

    else:

        display(result)


Recommendations for: Sugar
No direct rule found for Sugar

Recommendations for: Rice
No direct rule found for Rice

Recommendations for: Noodles
No direct rule found for Noodles
